# Dealing with Conflicts

This file shows how `Lark` deals with *shift-reduce* and *reduce-reduce* conflicts.
The following grammar is *ambiguous* because it does not specify the precedence of the arithmetical operators:
```
    expr : expr "+" expr
         | expr "*" expr
         | NUMBER
```

## Specification of the Scanner and the Parser

The operators `"+"` and `"*"` occur as *anonymous terminals*: they are given as string literals
inside the grammar rules.  `Lark` invents the names `PLUS` and `STAR` for these terminals and these
invented names will show up in the messages discussed below.

The expression following the arrow `->` is an *alias* for the corresponding alternative.  We will use
these aliases later in order to build an abstract syntax tree.  Aliases only change the *name* of the
node that is created in the parse tree, they have no influence on the parse table.

In [ ]:
grammar = r"""
    expr : expr "+" expr   -> add
         | expr "*" expr   -> mul
         | NUMBER          -> number

    NUMBER : /0|[1-9][0-9]*/

    %import common.WS
    %ignore WS
"""

## Making `Lark` Talk About Its Conflicts

By default, `Lark` is quite tight-lipped about shift-reduce conflicts: the parser is built, the conflicts
are resolved silently, and nothing is printed.  There are two reasons for this silence:

1. The conflict messages are written to the logger object `lark.logger` and the level of this logger is
   set to `logging.CRITICAL` when `Lark` is imported.  Hence, warnings are discarded.
2. The messages are only issued with the severity `warning` if the keyword argument `debug=True` is given
   when the parser is created.  Otherwise, they are issued with the severity `debug`.

Therefore, we have to lower the level of the logger *and* create the parser with `debug=True`.

In [ ]:
import logging
from contextlib import contextmanager
from lark import Lark, Transformer, logger

@contextmanager
def show_lark_warnings():
    old_level = logger.level
    logger.setLevel(logging.DEBUG)
    try:
        yield
    finally:
        logger.setLevel(old_level)

The keyword argument `parser='lalr'` is essential.  The *default* parser of `Lark` is an *Earley* parser,
and an Earley parser has no problem at all with an ambiguous grammar: it would simply return one of the
possible parse trees without complaining.

In [ ]:
with show_lark_warnings():
    parser = Lark(grammar, start='expr', parser='lalr', debug=True)

Every one of these messages names

1. the terminal for which the entry of the action table is ambiguous, and
2. the grammar rule that could have been used to reduce the symbol stack in this situation.

Note the parenthesized remark `(resolving as shift)`:  `Lark` resolves **every** shift-reduce conflict in
favour of shifting.  

The only thing we can do is to declare that we do not want to tolerate any conflicts at all.  If the parser
is created with the keyword argument `strict=True`, then the first shift-reduce conflict raises an exception
of class `GrammarError`.  It is a good idea to develop a grammar with `strict=True`, for then a conflict
cannot be overlooked.

In [ ]:
from lark import GrammarError

try:
    Lark(grammar, start='expr', parser='lalr', strict=True)
except GrammarError as e:
    print(e)

## Inspecting the LALR States

If the parser is created with `debug=True`, the parse table is kept in a form where the states are still the
*sets of marked rules* that we have discussed in the lecture.  The function `dump_states` below prints
these states.

The states of this table are *frozen sets of marked rules*.  They are not numbered, so the dictionary
`number` assigns a number to every state in order to make the output readable.  Since the states are sets,
iterating over them is not reproducible from one run to the next.  Therefore, the states are sorted before
they are numbered and the start state is put first, so that the numbering agrees with the convention used
by `Ply`.

In [ ]:
from lark.parsers.lalr_analysis import Shift

def dump_states(lark_parser):
    table  = lark_parser.parser.parser.parser.parse_table
    key    = lambda state: sorted(str(rule) for rule in state)
    start  = next(iter(table.start_states.values()))
    rest   = sorted((s for s in table.states if s != start), key=key)
    order  = [start] + rest
    number = { state: i for i, state in enumerate(order) }
    for state in order:
        print(f'state {number[state]}')
        for marked_rule in sorted(state, key=str):
            print(f'    {marked_rule}')
        print()
        for token, (action, arg) in sorted(table.states[state].items()):
            if action is Shift:
                print(f'    {token:<8} shift and go to state {number[arg]}')
            else:
                print(f'    {token:<8} reduce using rule {arg}')
        print()

Let us look at the action table that is generated.  Note that `Lark` marks the position of the parser
inside a rule with the character `*` instead of the bullet `•` that we use in the lecture notes, and that
the end of the input is called `$END`.

In the states that contain a marked rule of the form `<expr : expr PLUS expr * >` or
`<expr : expr STAR expr * >` there are two shift-reduce conflicts each.  All of them are resolved in favour
of shifting: the only reduce action that survives in these states is the one for `$END`.  The discarded
reduce actions are *not* shown here; in order to see them we have to read the warnings that were printed
above.

In [ ]:
dump_states(parser)

## Building an Abstract Syntax Tree

The parse tree that is returned by `Lark` is an object of class `Tree`.  In order to reuse the function
`tuple2dot`, we convert this tree into a nested tuple.  This is done with a `Transformer`:  for every
alias that occurs in the grammar, the transformer provides a method of the same name.  This method receives
the list of the children that have already been transformed and returns the value that should replace the
node.

In [ ]:
%run ../AST2Dot.ipynb

In [ ]:
class ASTBuilder(Transformer):
    def add(self, children):
        lhs, rhs = children
        return ('+', lhs, rhs)

    def mul(self, children):
        lhs, rhs = children
        return ('*', lhs, rhs)

    def number(self, children):
        return int(children[0])

The function `test(s)` takes a string `s` as its argument and tries to parse this string.  If all goes well,
an abstract syntax tree is returned and displayed.  If the string can't be parsed, `Lark` raises an
exception of class `UnexpectedInput`, which we catch in order to print a readable error message.

In [ ]:
from lark.exceptions import UnexpectedInput

def test(s):
    try:
        parse_tree = parser.parse(s)
    except UnexpectedInput as e:
        print(f'Syntax error:\n{e}')
        return None
    t = ASTBuilder().transform(parse_tree)
    d = tuple2dot(t)
    display(d)
    return t

The next example shows that this parser does not produce the abstract syntax that reflects the precedences
of the arithmetical operators:  the string `2*3+4` is parsed as `2*(3+4)`.

In [ ]:
test('2*3+4')

Due to the fact that all shift-reduce conflicts are resolved in favour of shift, all operators have the
same precedence and effectively associate to the right.

In [ ]:
test('2+3+4')

In [ ]:
test('1+2*3')

The notebook [02-Conflicts-Resolved.ipynb](02-Conflicts-Resolved.ipynb) shows how these conflicts can be
removed.  Since `Lark` does not support operator precedence declarations, the precedences and
associativities of the operators have to be encoded into the grammar by introducing one syntactical
variable per precedence level.